# How charts lie (and how not to get fooled)

_Data Wrangling_

**A chart can tell the truth about the numbers and still mislead the eye — here are the classic tricks and the honest fix for each.**

A chart is a visual encoding: it maps numbers to lengths, positions, angles, areas,
       and colours so your eye can compare them fast. The encoding can be faithful &mdash;
       the visual size of a thing matches its real size &mdash; or it can be broken so the
       visual impression no longer matches the numbers.

       The slippery part is that a misleading chart usually contains no false numbers.
       Every value is correct. What is wrong is the mapping: a truncated axis, an arbitrary
       second scale, a radius standing in for an area. The lie lives in the geometry, not the data,
       which is exactly why it is so easy to do by accident and so easy to miss.

---

This notebook is a practice scaffold for the **How charts lie (and how not to get fooled)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1) TRUNCATED Y-AXIS (lie) vs ZERO BASELINE (honest)
#    Four groups whose means are genuinely CLOSE.
# ============================================================
groups = ['Q1', 'Q2', 'Q3', 'Q4']
means  = [12.59, 13.39, 14.72, 15.82]   # max/min ratio = 1.26  -> nearly equal

fig, (ax_lie, ax_ok) = plt.subplots(1, 2, figsize=(10, 4))

# --- MISLEADING: baseline chopped up to 12, differences fill the frame ---
ax_lie.bar(groups, means, color='#ff7b72')
ax_lie.set_ylim(12, 16)                  # <-- the trick: axis does NOT start at 0
ax_lie.set_title('MISLEADING: y-axis truncated at 12\n(bars look ~6x apart)')

# --- HONEST: a bar encodes length, so the axis must start at 0 ---
ax_ok.bar(groups, means, color='#7ee787')
ax_ok.set_ylim(0, 16)                     # <-- honest baseline
ax_ok.set_title('HONEST: y-axis starts at 0\n(bars are nearly equal -- the truth)')

plt.tight_layout()
plt.savefig('truncated_vs_zero.png', dpi=120)

# ============================================================
# 2) DUAL Y-AXES (fake correlation) vs TWO HONEST PANELS
# ============================================================
t       = np.arange(12)
ad_spend = np.array([10, 11, 9, 12, 14, 13, 15, 16, 14, 17, 19, 18.])
sales    = np.array([200, 150, 260, 170, 210, 240, 190, 230, 280, 220, 250, 300.])
# sales is basically noise; the dual axis will MAKE it look like it tracks spend.

fig2, (ax_bad, ax_g1) = plt.subplots(1, 2, figsize=(11, 4))

# --- MISLEADING: two arbitrary scales tuned until the lines "coincide" ---
ax_bad.plot(t, ad_spend, color='#4ea1ff', label='ad spend')
ax_bad.set_ylabel('ad spend', color='#4ea1ff')
ax_twin = ax_bad.twinx()                  # <-- second y-axis = the trick
ax_twin.plot(t, sales, color='#ff7b72', label='sales')
ax_twin.set_ylabel('sales', color='#ff7b72')
ax_bad.set_title('MISLEADING: dual y-axes\n"sales tracks spend!" (it does not)')

# --- HONEST: two stacked panels, each on its own zeroed scale ---
gs = ax_g1.get_gridspec()
ax_g1.remove()
sub = fig2.add_gridspec(2, 1, left=0.55, right=0.98, hspace=0.5)
p1 = fig2.add_subplot(sub[0]); p1.plot(t, ad_spend, color='#4ea1ff'); p1.set_ylim(0, 20); p1.set_title('ad spend')
p2 = fig2.add_subplot(sub[1]); p2.plot(t, sales, color='#ff7b72');    p2.set_ylim(0, 320); p2.set_title('sales')
# The real correlation, reported as a number instead of implied by overlay:
r = np.corrcoef(ad_spend, sales)[0, 1]
print(f'actual correlation(ad_spend, sales) = {r:.2f}')   # weak -- no real tracking

plt.savefig('dual_axis_vs_panels.png', dpi=120)
print('saved truncated_vs_zero.png and dual_axis_vs_panels.png')

## Visualize the data & results

_The SAME four real group means from load_breast_cancer, drawn two ways: a truncated-axis bar chart (differences look huge) vs a zero-baseline bar chart (differences look small). Which one tells the truth?_

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer(as_frame=True)
df = d.frame                                   # 569 real tumor measurements

# Split rows into 4 equal-count groups by 'mean texture', then take the
# mean of 'mean radius' in each group -> four genuinely CLOSE numbers.
df['tq'] = pd.qcut(df['mean texture'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
means = df.groupby('tq', observed=True)['mean radius'].mean()
print(means.round(2).tolist())                 # -> [12.59, 13.39, 14.72, 15.82]

# True spread: tallest / shortest as the eye SHOULD read it from zero.
print('honest ratio max/min =', round(means.max() / means.min(), 2))   # 1.26

# Now truncate the axis at 12: the eye reads (value - 12) instead.
base = 12.0
drawn = (means - base)
print(drawn.round(2).tolist())                 # -> [0.59, 1.39, 2.72, 3.82]
print('truncated ratio max/min =', round(drawn.max() / drawn.min(), 2))  # 6.47
# 1.26x of real difference rendered as a 6.5x cliff -- pure axis trick.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate's slide shows two bars for last-quarter revenue: one looks roughly three times taller than the other, suggesting a huge gap. You notice the y-axis starts at $9.0\text{M}$ and the values are $\$9.2\text{M}$ and $\$9.8\text{M}$. What is going on, and how do you redraw it honestly?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the true ratio of the values: $9.8/9.2 \approx 1.07$. — _The real difference is about $7\%$ — small._
- Compute the drawn-height ratio with baseline $b=9.0$: heights are $0.2$ and $0.8$, ratio $0.8/0.2 = 4$. — _The truncated baseline makes a $7\%$ gap look like a $4\times$ gap — that is the exaggeration._
- Redraw the bar chart with the y-axis starting at $0$. — _A bar encodes a quantity by length, which is only meaningful from zero; the two bars will then look nearly equal, as they truly are._

**Answer:** The chart uses a truncated y-axis (baseline at $9.0\text{M}$), so the drawn bar lengths are $0.2$ and $0.8$ &mdash; a $4\times$ visual ratio &mdash; while the true values $9.2$ and $9.8$ differ by only about $7\%$. Fix: start the bar axis at $0$. The bars then look nearly identical, matching reality. (If you want to emphasise the small change, show it as a labelled percentage, not by chopping the axis.)

</details>

**Problem 2.** A figure overlays "ad spend" and "sales" as two lines on a chart with two y-axes, and they track each other beautifully. The caption says "advertising drives sales." Name the two separate problems and the honest redraw.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot the dual y-axes: each line has its own vertical scale, chosen by the author. — _By sliding and stretching one scale you can make almost any two upward series appear to coincide — the visual "match" is an artifact of arbitrary scaling._
- Spot the causal claim attached to a co-movement. — _Even a real correlation cannot show causation from a chart; a lurking variable (e.g. holiday season lifting both) or reverse causation fits equally well._
- Redraw as two stacked panels sharing the x-axis, or as a scatter of sales vs spend, and report the actual correlation; reword the caption to "associated with". — _Honest panels remove the scaling trick, and "associated" reserves causal language for an experiment._

**Answer:** Two problems: (1) dual y-axes &mdash; two arbitrary scales were tuned until the lines visually coincide, so the "match" is manufactured, not measured; (2) correlation framed as causation &mdash; a chart of co-movement cannot establish that spend drives sales. Fix: plot the two series in two stacked panels with a shared x-axis (or a scatter with the reported correlation), and caption it as an association, not a cause.

</details>

**Problem 3.** A report compares two regions with a single bar of the mean response time for each: both are about $200\text{ms}$, so the report concludes "the regions perform identically." Why might this be misleading, and what should be plotted instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognise that a bar of means discards all information about spread. — _Two groups can share a mean while one is tight around $200\text{ms}$ and the other swings from $50$ to $600\text{ms}$._
- Ask what the decision actually cares about — often the tail (the slow requests), not the average. — _A heavy upper tail means a bad experience for many users even when the mean looks fine._
- Plot the full distribution: a box plot, violin, or jittered points (or at least the median and the 95th percentile) per region. — _Showing spread reveals whether the regions truly behave the same, which the mean alone cannot._

**Answer:** It is misleading because aggregation hides variation: equal means can sit on top of very different spreads, and for latency the tail usually matters more than the average. Fix: show the distribution &mdash; box plot / violin / jittered points, or at minimum the median and the 95th percentile per region &mdash; instead of a single mean bar.

</details>